# Bharani's Retail & Co. — Phase 2: Data Cleaning & EDA

**Goal:** Clean the synthetic inventory dataset from Phase 1, then explore it —
replicating and extending the stock level / cost distribution / turnover analysis
from the Excel dashboard, but now in Python.

**Sections:**
1. Load & first look
2. Cleaning
3. Exploratory analysis
4. Save cleaned data for Phase 3

In [49]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_style('whitegrid')
np.random.seed(42)

## 1. Load & first look

In [50]:
df = pd.read_csv('../data/raw/inventory_data.csv')

print(f"Shape: {df.shape}")
df.head()

Shape: (50, 8)


,product_id,product_name,category,supplier,unit_cost,unit_price,stock_quantity,units_sold_last_month
0,P001,Yoga Mat,Electronics,Peak Distributors,40.51,50.47,52,173
1,P002,Action Figure,Electronics,Summit Wholesale,9.61,37.17,119,129
2,P003,Sketch Pad,Electronics,Orbit Traders,108.82,213.38,214,56
3,P004,Highlighter Pack,Stationery,Peak Distributors,122.37,11.88,412,40
4,P005,Water Bottle,Toys,Peak Distributors,27.54,287.59,172,26


In [51]:
# Column types, non-null counts — spot obvious issues fast
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   product_id             50 non-null     str    
 1   product_name           50 non-null     str    
 2   category               50 non-null     str    
 3   supplier               50 non-null     str    
 4   unit_cost              50 non-null     float64
 5   unit_price             50 non-null     float64
 6   stock_quantity         50 non-null     int64  
 7   units_sold_last_month  50 non-null     int64  
dtypes: float64(2), int64(2), str(4)
memory usage: 3.3 KB


In [52]:
# Quick summary stats for numeric columns
df.describe()

,unit_cost,unit_price,stock_quantity,units_sold_last_month
count,50.000000,50.00000,50.00000,50.00000
mean,73.838800,145.52620,254.88000,116.10000
std,45.478669,93.78442,137.26626,57.61032
min,5.310000,11.88000,1.00000,3.00000
25%,33.852500,54.67750,144.25000,66.00000
50%,63.550000,136.30500,271.50000,131.00000
75%,110.415000,220.20500,355.00000,165.00000
max,149.300000,299.79000,477.00000,200.00000


## 2. Cleaning

Checklist to work through (delete what doesn't apply, add what does once you see your real columns):
- Missing values
- Duplicate rows
- Inconsistent text (casing, whitespace, typos in category names)
- Data types (dates as strings, numbers as objects, etc.)
- Outliers / impossible values (negative stock, negative price)

In [53]:
# Missing values — count and %
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct}).query('missing_count > 0')

,missing_count,missing_pct


In [54]:
# Duplicate rows
print(f"Duplicate rows: {df.duplicated().sum()}")
# df = df.drop_duplicates()

Duplicate rows: 0


In [55]:
# Example: standardize text columns — adjust column names to match your CSV
# for col in ['category', 'supplier', 'warehouse']:
#     if col in df.columns:
#         df[col] = df[col].str.strip().str.title()

In [56]:
# Example: fix data types — adjust to match your columns
# df['date_received'] = pd.to_datetime(df['date_received'], errors='coerce')
# df['unit_cost'] = pd.to_numeric(df['unit_cost'], errors='coerce')

In [57]:
# Sanity checks for impossible values
# print("Negative stock:", (df['stock_qty'] < 0).sum())
# print("Negative cost:", (df['unit_cost'] < 0).sum())

## 3. Exploratory analysis

Mirroring the Excel dashboard: stock levels, cost distribution, turnover.

In [58]:
# Stock levels by category
# df.groupby('category')['stock_qty'].sum().sort_values(ascending=False).plot(kind='bar', figsize=(10,5), title='Stock by Category')
# plt.ylabel('Units in stock')
# plt.show()

In [59]:
# Cost distribution
# plt.figure(figsize=(10,5))
# sns.histplot(df['unit_cost'], bins=30, kde=True)
# plt.title('Unit Cost Distribution')
# plt.show()

In [60]:
# Turnover rate (needs sales/usage data alongside stock — adjust formula to your columns)
# df['turnover_rate'] = df['units_sold'] / df['avg_stock_qty']
# df.sort_values('turnover_rate', ascending=False)[['product_name','turnover_rate']].head(10)

## 4. Save cleaned data for Phase 3

In [61]:
df.to_csv('../data/processed/inventory_clean.csv', index=False)
print("Saved cleaned dataset.")

Saved cleaned dataset.


In [62]:
df.columns

Index(['product_id', 'product_name', 'category', 'supplier', 'unit_cost',
       'unit_price', 'stock_quantity', 'units_sold_last_month'],
      dtype='str')

In [63]:
# Turnover Summary
df['turnover_rate'] = df['units_sold_last_month'] / df['stock_quantity']

turnover_summary = df.groupby('category')['turnover_rate'].mean().sort_values(ascending=False)
print("Average turnover rate by category:")
print(turnover_summary)

Average turnover rate by category:
category
Toys              37.358568
Sportswear         5.125851
Electronics        1.375757
Stationery         0.547733
Home & Kitchen     0.355803
Name: turnover_rate, dtype: float64
